In [1]:
import pandas as pd
import uuid

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
PPG_dict = {
    'PPG-CCMC': '55134',
    'PPG-Mat': '55135',
    'PPG-PROFMAT': '55136',
    'PPG-MECAI': '55137',
    'PPG-PIPGEs': '104131',
    # Sem o PPG (27/04/2026)
    'CCMC': '55134',
    'Mat': '55135',
    'PROFMAT': '55136',
    'MECAI': '55137',
    'PIPGEs': '104131'
}

def cria_link( codigo, curso ):
    curso = curso.strip()

    try:
        ppg = PPG_dict[ curso ]
        return f'https://www.icmc.usp.br/pos-graduacao/disciplinas?programa={ppg}&disciplina={codigo}'
    except KeyError:
        return f'https://uspdigital.usp.br/jupiterweb/obterDisciplina?nomdis=&sgldis={codigo}'


In [4]:
df = pd.read_excel('./2026/2/Elenco_SME_2026-2-ICMC-em elaboração-enviado graduação.xlsx', skiprows = 2)

In [5]:
df = df.dropna(subset=["Disciplina (código)"])
df = df.fillna( '' )

In [6]:
# Encontra todas as colunas originais que começam com 'Horário '
horario_cols_original = [col for col in df.columns if col.startswith('Horário ')]

# Cria os novos nomes (ex: 'horario1', 'horario2', ...)
horario_cols_new = [f'horario{i+1}' for i in range(len(horario_cols_original))]

# Cria o mapa de renomeação (ex: {'Horário 1': 'horario1', ...})
horario_rename_map = dict(zip(horario_cols_original, horario_cols_new))

print(f"Colunas de horário detectadas: {len(horario_cols_original)}")
print(f"Mapa de renomeação: {horario_rename_map}")

Colunas de horário detectadas: 4
Mapa de renomeação: {'Horário 1': 'horario1', 'Horário 2': 'horario2', 'Horário 3': 'horario3', 'Horário 4': 'horario4'}


In [7]:
# Começa com o mapa de renomeação padrão
rename_columns_map = {
    "Disciplina (código)": "codigo",
    "Disciplina (nome completo)": "nome",
    "Curso(s)": "curso",
    "Turma": "turma"
}

# Adiciona dinamicamente as colunas de horário ao mapa
rename_columns_map.update(horario_rename_map)

# Renomeia usando o mapa combinado
df = df.rename(columns=rename_columns_map, errors='ignore')

# O restante da célula (drop) permanece o mesmo
df = df.drop( ['observações', 'Docente\n(nome completo sem abreviações)', 'NUSP', 
               'Utilizará laboratório?\n(sim ou não)', 'Sala \n(a definir)', 'Será espelho com Pós?', 'Deve ser alocada no ICMC?\n', 
               'créditos' ], axis = 1, errors='ignore' )

In [8]:
print("Iniciando validação de duplicatas...")

# 1. Trata a coluna 'turma' como você faz na Célula 14
# Converte para numérico ('' vira NaN), preenche NaN com 1.0, e converte para int
df['turma'] = pd.to_numeric(df['turma'], errors='coerce').fillna(1.0).astype(int)

# 2. Cria a coluna de ID composta
df['id_composto'] = df['codigo'].astype(str) + ',' + df['turma'].astype(str)

# 3. Encontra IDs duplicados
duplicated_ids = df[df.duplicated(subset=['id_composto'], keep=False)]

# 4. Se houver duplicatas, levanta um erro claro e para o script
if not duplicated_ids.empty:
    print("\\n--- ERRO: DISCIPLINAS DUPLICADAS ENCONTRADAS ---")
    print("As seguintes combinações de 'codigo' e 'turma' estão duplicadas no arquivo Excel:")
    print(duplicated_ids[['codigo', 'nome', 'turma', 'id_composto']].sort_values(by='id_composto'))
    print("\\nPor favor, corrija o arquivo Excel para garantir que cada par (código, turma) seja único.")
    print("A duplicidade pode causar falhas na busca de dados e na plataforma final.")

    # Gera uma exceção para parar a execução do notebook
    raise ValueError("Disciplinas duplicadas encontradas. Verifique o output acima e corrija o Excel.")
else:
    print("✓ Validação concluída. Nenhum ID duplicado encontrado.")

# Renomeia a coluna para 'id' para uso posterior
df = df.rename(columns={'id_composto': 'id'})

Iniciando validação de duplicatas...
✓ Validação concluída. Nenhum ID duplicado encontrado.


In [9]:
# Preencher dados Ementa e Nível (P ou G)
ementa = []
nivel = []
for codigo, curso in zip( df[ 'codigo' ], df[ 'curso' ] ):
    ementa.append( cria_link( codigo, curso ) )
    try:
        p = PPG_dict[ curso ]
        nivel.append( 'p' )
    except KeyError:
        nivel.append( 'g' )

df[ 'ementa' ] = ementa
df[ 'nivel' ] = nivel

In [10]:
df

,codigo,nome,curso,turma,horario1,horario2,horario3,horario4,id,ementa,nivel
0,SME0110,Programação Matemática,BCC,1,Segunda - 08:10 / 09:50,Quarta - 10:10 / 11:50,,,"SME0110,1",https://uspdigital.usp.br/jupiterweb/obterDisc...,g
1,SME0110,Programação Matemática,BCC,2,Segunda - 10:10 / 11:50,Quarta - 08:10 / 09:50,,,"SME0110,2",https://uspdigital.usp.br/jupiterweb/obterDisc...,g
2,SME0123,Estatística,BCC,1,Terça - 14:20 / 16:00,Sexta - 08:10 / 09:50,,,"SME0123,1",https://uspdigital.usp.br/jupiterweb/obterDisc...,g
3,SME0123,Estatística,BCC,2,Terça - 16:20 / 18:00,Sexta - 10:10 / 11:50,,,"SME0123,2",https://uspdigital.usp.br/jupiterweb/obterDisc...,g
4,SME0142,Algebra Linear e Aplicações,"BCC, BCDados",1,Terça - 10:10 / 11:50,Quinta - 10:10 / 11:50,,,"SME0142,1",https://uspdigital.usp.br/jupiterweb/obterDisc...,g
...,...,...,...,...,...,...,...,...,...,...,...
70,EST5804,Tópicos Avançados de Pesquisa I (Louzada).,PIPGEs,1,,,,,"EST5804 ,1",https://www.icmc.usp.br/pos-graduacao/discipli...,p
71,SME0814,Projeto Supervisionado em Estatística I,BECD,1,,,,,"SME0814 ,1",https://uspdigital.usp.br/jupiterweb/obterDisc...,g
72,SME0815,Projeto Supervisionado em Estatística II,BECD,1,,,,,"SME0815 ,1",https://uspdigital.usp.br/jupiterweb/obterDisc...,g
73,SME0880,Projeto de Graduação em Estatística I,BECD,1,,,,,"SME0880 ,1",https://uspdigital.usp.br/jupiterweb/obterDisc...,g


In [11]:
json_data = df.to_dict(orient='records')

In [12]:
for item in json_data:
    keys_to_remove = [k for k, v in item.items() if pd.isna(v) and k == "horario2"]
    for k in keys_to_remove:
        del item[k]

In [13]:
len(json_data)

75

In [14]:
dias_map = {
    "Segunda": "Seg.",
    "Terça": "Ter.",
    "Quarta": "Qua.",
    "Quinta": "Qui.",
    "Sexta": "Sex.",
    "Sábado": "Sáb.",
    "Domingo": "Dom."
}

In [15]:
def processar_horario(horario):
    # Verifica se é NaN, None ou uma string vazia/só com espaços
    if pd.isna(horario) or not horario or horario.strip() == "":
        return None
    try:
        # Exemplo: "Terça - 14:20 / 16:00"
        dia_e_horas = horario.split('-')
        dia_extenso = dia_e_horas[0].strip()
        dia = dias_map.get(dia_extenso, dia_extenso)  # Se não tiver no mapa, mantém original
        horas = dia_e_horas[1].strip().split('/')
        inicio = horas[0].strip()
        fim = horas[1].strip()
        return {"dia": dia, "inicio": inicio, "fim": fim}
    except Exception as e:
        print(f"Erro ao processar: {horario} -> {e}")
        return None

In [16]:
for item in json_data:
    horarios = []

    # USA A LISTA DINÂMICA 'horario_cols_new'
    for key in horario_cols_new:
        if key in item:
            h = processar_horario(item[key]) # Sua função já trata NaNs
            if h:
                horarios.append(h)
            del item[key]  # Remove os campos antigos (ex: 'horario1', 'horario2')

    item['horarios'] = horarios

    # Aproveitar que já está sendo iterado por todas as turmas
    item['turma'] = int(item['turma'])
    item['codigo'] = str(item['codigo'])

In [17]:
def hora_para_minutos(hora_str):
    """Converte 'HH:MM' para minutos desde 00:00"""
    h, m = map(int, hora_str.split(":"))
    return h * 60 + m

In [18]:
def horarios_conflitam(h1, h2):
    """Verifica se dois horários conflitam no mesmo dia"""
    if h1['dia'] != h2['dia']:
        return False

    inicio1 = hora_para_minutos(h1['inicio'])
    fim1 = hora_para_minutos(h1['fim'])
    inicio2 = hora_para_minutos(h2['inicio'])
    fim2 = hora_para_minutos(h2['fim'])

    return max(inicio1, inicio2) < min(fim1, fim2)

In [19]:
# Inicializa os sets de conflitos
for turma in json_data:
    turma['conflitos'] = set()

# Compara todas as turmas em pares (sem repetições)
for i in range(len(json_data)):
    turma_i = json_data[i]
    for j in range(i + 1, len(json_data)):
        turma_j = json_data[j]

        # Verifica se algum horário da turma_i conflita com algum da turma_j
        for h1 in turma_i.get('horarios', []):
            for h2 in turma_j.get('horarios', []):
                if horarios_conflitam(h1, h2):
                    turma_i['conflitos'].add(turma_j['id'])
                    turma_j['conflitos'].add(turma_i['id'])
                    break  # Um conflito já basta

# Converte os sets para listas (JSON serializável)
for turma in json_data:
    turma['conflitos'] = list(turma['conflitos'])

In [20]:
def marcar_turmas_noturnas(json_data):
    for turma in json_data:
        turma['noturna'] = False  # valor padrão
        for horario in turma.get('horarios', []):
            try:
                hora, minuto = map(int, horario['inicio'].split(':'))
                if hora >= 18:
                    turma['noturna'] = True
                    break  # Já é noturna, não precisa verificar os outros horários
            except Exception as e:
                print(f"Erro ao processar horário {horario['inicio']} da turma {turma.get('id')}: {e}")


In [21]:
marcar_turmas_noturnas(json_data)

* Buscar nas páginas do ICMC da Graduação e da Pós para identificar qual o nível da disciplina, preencher a ementa com o link correto e buscar a quantidade de créditos

```
Pós - f'https://www.icmc.usp.br/pos-graduacao/disciplinas?programa={ppg}&disciplina={codigo}'
Graduação - f'https://uspdigital.usp.br/jupiterweb/obterDisciplina?nomdis=&sgldis={codigo}'
```

In [22]:
import time

In [ ]:
try:
    driver = webdriver.Chrome()
except Exception as e:
    print(f"Erro ao inicializar o WebDriver: {e}")
    print("Verifique se o Google Chrome está instalado.")
    exit()

dados_coletados_web = {}

print("Iniciando a busca por disciplinas...")

iteracao = 0
for turma in json_data:
    codigo = turma['codigo'].strip()
    disciplina_encontrada = False
    print(f"\n🔎 Processando disciplina: {codigo}")

    # TENTATIVA NA GRADUAÇÃO
    url_graduacao = f'https://uspdigital.usp.br/jupiterweb/obterDisciplina?nomdis=&sgldis={codigo}'
    driver.get(url_graduacao)
    time.sleep(1) # Pequena pausa para garantir o carregamento da página

    try:
        # Verifica se a div de erro existe
        driver.find_element(By.ID, "web_mensagem")
        print(f"   - '{codigo}' não encontrada na Graduação. Verificando na Pós-Graduação...")

    except NoSuchElementException:
        # Se a div de erro NÃO existe, a disciplina é da graduação.
        print(f"   ✓ '{codigo}' encontrada na Graduação!")
        try:
            # Extrai o nome da disciplina
            nome_completo_tag = driver.find_element(By.XPATH, "//span[contains(., 'Disciplina:')]/b")
            # O texto é "Disciplina: SME0104 - Cálculo Numérico", pegamos só o nome
            nome_disciplina = nome_completo_tag.text.split('-', 1)[1].strip()

            # Extrai os créditos-aula
            # A busca por XPath é mais robusta:
            # Encontre o <b> com o texto 'Créditos Aula:', suba para a célula <td>, e pegue a próxima célula <td>
            creditos_aula_tag = driver.find_element(By.XPATH, "//b[contains(text(), 'Créditos Aula:')]/ancestor::td/following-sibling::td")
            creditos_aula = creditos_aula_tag.text.strip()

            tipo_disciplina_tag = driver.find_element(By.XPATH, "//b[contains(text(), 'Tipo:')]/ancestor::td/following-sibling::td")
            tipo_disciplina = tipo_disciplina_tag.text.strip()

            dados_coletados_web[codigo] = {
                'codigo': codigo,
                'nome': nome_disciplina,
                'nivel': 'g',
                'creditos': int(creditos_aula),
                'ementa_link': url_graduacao,
                'tipo': tipo_disciplina
            }
            disciplina_encontrada = True

        except Exception as e:
            print(f"      -> Erro ao extrair dados da página da Graduação para '{codigo}': {e}")
            dados_coletados_web[codigo] = {
                'codigo': codigo,
                'status': 'Erro na Extração (Graduação)',
                'url': url_graduacao
            }

    # TENTATIVA NA PÓS-GRADUAÇÃO (se não foi encontrada na graduação)
    if not disciplina_encontrada:
        for ppg in PPG_dict.values():
            url_pos = f'https://www.icmc.usp.br/pos-graduacao/disciplinas?programa={ppg}&disciplina={codigo}'
            driver.get(url_pos)
            time.sleep(1)

            try:
                # Seletor ATUALIZADO para o nome da disciplina na Pós
                nome_disciplina_pos = driver.find_element(By.CSS_SELECTOR, 'div.page-header h3').text.strip()
                
                # Seletor ATUALIZADO para o número de créditos na Pós
                creditos_p_tag = driver.find_element(By.XPATH, "//p[b[contains(text(), 'Nº de créditos:')]]")
                # Extrai o texto "Nº de créditos: 12", divide no ":" e pega a segunda parte
                creditos_pos = creditos_p_tag.text.split(':')[1].strip()

                duracao_element = driver.find_element(By.XPATH, "//tbody/tr[2]/td[count(//tbody/tr[1]/td[contains(., 'Duração')]/preceding-sibling::td) + 1]")
                
                duracao_valor = duracao_element.text

                print(f"   ✓ '{codigo}' encontrada na Pós-Graduação (Programa: {ppg})!")
                dados_coletados_web[codigo] = {
                    'codigo': codigo,
                    'nome': nome_disciplina_pos,
                    'nivel': 'p',
                    'creditos': int(creditos_pos),
                    'ementa_link': url_pos,
                    'tipo': "Semestral" if int(duracao_valor.split(" ")[0]) > 8 else "Bimestral" 
                }
                disciplina_encontrada = True
                break # Para o loop dos PPGs, pois já encontramos a disciplina
            except NoSuchElementException:
                print(f"   - '{codigo}' não encontrada no programa de Pós '{ppg}'.")
                continue

    # 3. Se não foi encontrada em nenhum lugar
    if not disciplina_encontrada:
        print(f"   ❌ '{codigo}' não encontrada em nenhuma das fontes.")
        dados_coletados_web[codigo] = {
            'codigo': codigo,
            'status': 'Não encontrada'
        }
    else: 
        print("Iteração: ", iteracao)

    iteracao+=1

# Fecha o navegador
driver.quit()

Iniciando a busca por disciplinas...

🔎 Processando disciplina: SME0110
   ✓ 'SME0110' encontrada na Graduação!
Iteração:  0

🔎 Processando disciplina: SME0110
   ✓ 'SME0110' encontrada na Graduação!
Iteração:  1

🔎 Processando disciplina: SME0123
   ✓ 'SME0123' encontrada na Graduação!
Iteração:  2

🔎 Processando disciplina: SME0123
   ✓ 'SME0123' encontrada na Graduação!
Iteração:  3

🔎 Processando disciplina: SME0142
   ✓ 'SME0142' encontrada na Graduação!
Iteração:  4

🔎 Processando disciplina: SME0142
   ✓ 'SME0142' encontrada na Graduação!
Iteração:  5

🔎 Processando disciplina: SME0206
   ✓ 'SME0206' encontrada na Graduação!
Iteração:  6

🔎 Processando disciplina: SME0211
   ✓ 'SME0211' encontrada na Graduação!
Iteração:  7

🔎 Processando disciplina: SME0212
   ✓ 'SME0212' encontrada na Graduação!
Iteração:  8

🔎 Processando disciplina: SME0213
   ✓ 'SME0213' encontrada na Graduação!
Iteração:  9

🔎 Processando disciplina: SME0220
   ✓ 'SME0220' encontrada na Graduação!
Iteração

In [24]:
TCC_ESTAGIO = (
    "Trabalho de Conclusão de Curso",
    "Estágio Supervisionado"
)

**Turmas com ERRO pois não existe no site**
```json
    [{'codigo': 'SME0231', 
    'nome': 'Análise Topológica de Dados ', 
    'curso': 'BMACC', 
    'turma': 1, 
    'Deve ser alocada no ICMC?\n': '', 
    'id': 'SME0231,1', 
    'ementa': 'https://uspdigital.usp.br/jupiterweb/obterDisciplina?nomdis=&sgldis=SME0231',
    'nivel': 'g', 
    'horarios': [], 
    'conflitos': [], 
    'noturna': False},
    {'codigo': 'MAI5014', 
    'nome': 'Metodologia de Pesquisa e Desenvolvimento ', 
    'curso': 'MECAI', 
    'turma': 1, 
    'Deve ser alocada no ICMC?\n': '', 
    'id': 'MAI5014,1', 
    'ementa': 'https://www.icmc.usp.br/pos-graduacao/disciplinas?programa=55137&disciplina=MAI5014 ', 
    'nivel': 'p', 
    'horarios': [], 
    'conflitos': [], 
    'noturna': False}]
```

In [25]:
for turma in json_data:
    if turma['codigo'] in ['SME0231', "MAI5014"]:
        continue
    dados = dados_coletados_web[turma['codigo']]
    turma['codigo'] = turma['codigo'].strip()
    try:
        turma['nome'] = dados['nome'].strip()
    except:
        print(turma)
        turma['carga'] = 1
        raise IndexError()
    turma['nivel'] = dados['nivel']
    # turma['créditos'] = int(dados['creditos'])
    turma['ementa'] = dados['ementa_link']

    turma['turma'] = 1 if not turma['turma'] else int(turma['turma'])

    # Calcular a carga didática

    # Graduação
    if(turma['nivel'] == 'g'):
        if any(termo in turma['nome'] for termo in TCC_ESTAGIO):
            turma['carga'] = 0.25
        else:
            turma['carga'] = 0.25 * int(dados['creditos'])

    # Pós
    elif(turma['nivel'] == 'p'):
        if(dados['creditos'] >= 3): 
            if(dados['tipo'] == "Semestral" ):
                turma['carga'] = 1.0
            else:
                turma['carga'] = 0.5
        else:
             turma['carga'] = 0


    # Gerar UUID
    turma['uuid'] = str(uuid.uuid4())

In [26]:
json_data.sort(key=lambda x: x['codigo'])

In [27]:
import json

In [ ]:
with open(r'public/turmas_novo.json', 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)

In [29]:
with open(r'./app/context/dados.ts', 'w', encoding='utf-8') as f:
    f.write('import { TurmaDataInicial } from "../types";\n')
    f.write('\nexport const turmasJson: TurmaDataInicial[] = ')
    json.dump(json_data, f, ensure_ascii=False, indent=2)